<h3 style="color:#6FA8DC; font-weight:bold">02 — Outlier Detection using Z-Score</h3>

This notebook focuses completely on the **Z-Score method** for detecting and handling outliers.

We will cover:
- What Z-Score means
- Formula and intuition
- Why `|Z| > 3` is commonly used
- Step-by-step calculation
- Detecting outliers using SciPy
- Detecting outliers manually with pandas
- Removing outliers
- Capping outliers
- Visualization before/after
- When Z-Score should and should not be used
- Train/test and production considerations

<h5 style="color:#78B89A; font-weight:bold;">What is Z-Score? → simple meaning</h5>

Z-Score tells us **how many standard deviations a value is away from the mean**.

Formula:

```text
Z = (x - mean) / standard deviation
```

Interpretation:

```text
Z = 0  → exactly at the mean
Z = 1  → 1 standard deviation above mean
Z = -2 → 2 standard deviations below mean
Z = 4  → very far above mean
```

A common rule is:

```text
|Z| > 3 → potential outlier
```

This is a **rule of thumb**, not a universal law.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.Series([10, 12, 11, 13, 12, 14, 15, 13, 11, 12, 100])

df = pd.DataFrame({"value": data})
df

<h5 style="color:#78B89A; font-weight:bold;">Calculate Z-Score manually → understand the formula</h5>

In [ ]:
mean = df["value"].mean()
std = df["value"].std()

df["z_score"] = (df["value"] - mean) / std

df

In [ ]:
df["is_outlier"] = df["z_score"].abs() > 3

df

<h5 style="color:#78B89A; font-weight:bold;">Using SciPy → practical ML workflow</h5>

`scipy.stats.zscore()` calculates the Z-Score for us.

In [ ]:
from scipy.stats import zscore

df["z_score_scipy"] = zscore(df["value"])

df

<h5 style="color:#78B89A; font-weight:bold;">Find only the outlier rows</h5>

In [ ]:
outliers = df[df["z_score"].abs() > 3]

print("Potential outliers:")
display(outliers)

<h5 style="color:#78B89A; font-weight:bold;">Remove outliers → when the values are confirmed errors</h5>

Do not remove observations just because Z-Score identifies them.

First determine whether the observation is:
- a genuine rare event → usually keep
- a data-entry/measurement error → investigate and possibly remove

In [ ]:
df_without_outliers = df[df["z_score"].abs() <= 3].copy()

print("Original rows:", len(df))
print("Rows after removal:", len(df_without_outliers))

df_without_outliers

<h5 style="color:#78B89A; font-weight:bold;">Capping using Z-Score limits → preserve rows</h5>

Instead of deleting rows, we can cap extreme observations.

```text
upper limit = mean + 3 × std
lower limit = mean - 3 × std
```

In [ ]:
lower_limit = mean - 3 * std
upper_limit = mean + 3 * std

df["value_capped"] = df["value"].clip(
    lower=lower_limit,
    upper=upper_limit
)

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

df[["value", "value_capped"]]

<h5 style="color:#78B89A; font-weight:bold;">Visualize before and after</h5>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=df["value"], ax=axes[0])
axes[0].set_title("Before Z-Score Treatment")

sns.boxplot(x=df["value_capped"], ax=axes[1])
axes[1].set_title("After Z-Score Capping")

plt.tight_layout()
plt.show()

<h5 style="color:#78B89A; font-weight:bold;">Z-Score on a more realistic dataset</h5>

Let's create a salary dataset with a few extreme observations.

In [ ]:
np.random.seed(42)

salary = np.concatenate([
    np.random.normal(50000, 8000, 200),
    [150000, 180000, 250000]
])

salary_df = pd.DataFrame({"salary": salary})

salary_df["z_score"] = zscore(salary_df["salary"])

salary_outliers = salary_df[salary_df["z_score"].abs() > 3]

print("Potential outliers:", len(salary_outliers))
display(salary_outliers)

<h5 style="color:#78B89A; font-weight:bold;">When should I use Z-Score?</h5>

Good starting point when:
- the numerical feature is approximately normally distributed
- extreme values are relatively rare
- mean and standard deviation are meaningful

Be careful when:
- the data is strongly skewed
- the distribution has heavy tails
- the feature naturally contains extreme values

For strongly skewed data, **IQR or percentile-based methods may be more appropriate**.

<h5 style="color:#78B89A; font-weight:bold;">Important production rule → learn limits only from training data</h5>

Do not calculate mean/std using the complete dataset before train-test splitting.

Correct idea:

```text
Training data
     ↓
Calculate mean + std
     ↓
Calculate Z-Score limits
     ↓
Apply same limits to validation/test/production data
```

Otherwise information from future/test data can leak into the preprocessing step.

In [ ]:
from sklearn.model_selection import train_test_split

X = salary_df[["salary"]]
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

train_mean = X_train["salary"].mean()
train_std = X_train["salary"].std()

lower = train_mean - 3 * train_std
upper = train_mean + 3 * train_std

X_train["z_outlier"] = (
    (X_train["salary"] < lower) |
    (X_train["salary"] > upper)
)

X_test["z_outlier"] = (
    (X_test["salary"] < lower) |
    (X_test["salary"] > upper)
)

print("Learned from training data:")
print("Lower:", lower)
print("Upper:", upper)

<h3 style="color:#6FA8DC; font-weight:bold">Z-Score Revision</h3>

```text
Z = (x - mean) / std

If |Z| > 3
       ↓
Potential outlier
       ↓
Investigate
   ↙       ↘
error    genuine
 ↓          ↓
remove    keep / treat
```

⭐ Remember:

**Z-Score is a detection rule, not an automatic instruction to delete data.**